In [1]:
import os
import re
import string

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from sklearn.metrics import(
    accuracy_score,
    classification_report,
    confusion_matrix
)

from collections import Counter


In [2]:
path = 'Datasets' 
csv_path = os.path.join(path,"IMDB Dataset.csv")

df = pd.read_csv(csv_path)

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
print(df.shape)

(50000, 2)


In [4]:
import re

email_pattern = r"[\w.-]+@[\w.-]+\.\w+"
url_pattern = r"https?://\S+\.\S"

URL = re.compile(url_pattern)
EMAIL = re.compile(email_pattern)

In [5]:
# Removal of Stop words

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

nltk.download("stopwords")
nltk.download("punkt")

stop_words = stopwords.words("english")


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/mukundanramesh/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mukundanramesh/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [6]:
def text_clean(cell):
  if not isinstance(cell,str):
    return " "

  cell = EMAIL.sub(" ",cell)
  cell = URL.sub(" ",cell)
  cell = cell.lower()
  cell = re.sub(r"\d+"," ",cell)
  cell = re.sub(r"[^\w\s]","",cell)

  cell = [word for word in cell.split() if word not in stop_words]
  cell = " ".join(cell)

  return cell

df["clean_review"] = df["review"].apply(text_clean)

df.head()

,review,sentiment,clean_review
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...


In [7]:
df["label"] = df["sentiment"].map(
    {
        "positive":1,
        "negative":0
    }
)

df.head()

,review,sentiment,clean_review,label
0,One of the other reviewers has mentioned that ...,positive,one reviewers mentioned watching oz episode yo...,1
1,A wonderful little production. <br /><br />The...,positive,wonderful little production br br filming tech...,1
2,I thought this was a wonderful way to spend ti...,positive,thought wonderful way spend time hot summer we...,1
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake thinks...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stunni...,1


In [8]:
train_texts, test_texts, train_labels,test_labels = train_test_split(
    df["clean_review"].values,
    df["label"].values,
    test_size = 0.2,
    random_state = 42
)

print("Train texts shape:",train_texts.shape)
print("Test texts shape:",test_texts.shape)

Train texts shape: (40000,)
Test texts shape: (10000,)


In [9]:
# Build Vocab

MAX_VOCAB_SIZE = 20000

word_counter = Counter()

for text in train_texts:
  words = text.split()
  word_counter.update(words)

most_common_words = word_counter.most_common(
    MAX_VOCAB_SIZE - 2
)

word_to_index = {
    "<PAD>":0,
    "<UNK>":1
}

for word, _ in most_common_words:
  word_to_index[word] = len(word_to_index)


print("Vocabulary size:",len(word_to_index))

Vocabulary size: 20000


In [11]:
# Tokenize and pad

MAX_LENGTH = 500

def encode_text(text):
  words = text.split()

  encode = [
      word_to_index.get(word,1) for word in words
  ]

  encode = encode[:MAX_LENGTH]

  if len(encode) < MAX_LENGTH:
    encode = encode + [0] * (MAX_LENGTH - len(encode))

  return encode

example = "this movie good"

print(encode_text(example)[:10])

[1, 3, 7, 0, 0, 0, 0, 0, 0, 0]


In [12]:
class IMDBDataset(Dataset):
  def __init__(self,texts,labels):
    self.texts = texts
    self.labels = labels

  def __len__(self):
    return len(self.texts)

  def __getitem__(self,idx):
    text = self.texts[idx]
    label = self.labels[idx]

    encoded_text = encode_text(text)
    return torch.tensor(encoded_text),torch.tensor(label)

train_dataset = IMDBDataset(train_texts,train_labels)
test_dataset = IMDBDataset(test_texts,test_labels)

In [13]:
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

In [14]:
# DEEP LSTM Model

class DeepLSTM(nn.Module):

  def __init__(
      self,
      vocab_size,
      embedding_dim = 512,
      hidden_dim = 512,
      num_layers = 5,
      dropout = 0.4
  ):

    super().__init__()

    # word embedding
    self.embedding = nn.Embedding(
        num_embeddings = vocab_size,
        embedding_dim = embedding_dim,
        padding_idx = 0
    )

    # LSTM layers
    self.lstm = nn.LSTM(
        input_size = embedding_dim,
        hidden_size = hidden_dim,
        num_layers = num_layers,
        batch_first = True,
        dropout = dropout,
    )


    self.dropout = nn.Dropout(dropout)

    # Fully connected layer
    self.fc = nn.Linear(hidden_dim,1)

  def forward(self,x):
    embeddings = self.embedding(x) # x- [64,250] -> [64,250,128]

    output,(hidden,cell) = self.lstm(embeddings) #

    hidden = self.dropout(hidden[-1]) #[64,128]

    output = self.fc(hidden) # [64,1]

    return output.squeeze(1)


In [15]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")

vocab_size = len(word_to_index)

model = DeepLSTM(vocab_size).to(device)

print(model)

DeepLSTM(
  (embedding): Embedding(20000, 512, padding_idx=0)
  (lstm): LSTM(512, 512, num_layers=5, batch_first=True, dropout=0.4)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=512, out_features=1, bias=True)
)


In [16]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=1e-4)

In [19]:
from tqdm import tqdm
def train_loop(model, train_loader, criterion, optimizer):
  model.train()

  running_loss = 0
  correct = 0
  total = 0

  for images,labels in tqdm(train_loader):
    images = images.to(device)
    labels = labels.to(device).float() # Cast labels to float

    # Prediction
    logits = model(images)

    # Compute the loss
    loss = criterion(logits,labels)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Weight update
    optimizer.step()

    running_loss += loss.item()


    predictions = torch.round(torch.sigmoid(logits))
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

  epoch_loss = running_loss / len(train_loader)
  epoch_acc = correct / total

  print(f"Train Loss : {epoch_loss:.4f} | Train Acc : {epoch_acc:.4f}")


device = torch.device("mps" if torch.mps.is_available() else "cpu")

for epoch in range(20):
  train_loop(model,train_loader,criterion,optimizer)

100%|████████████████████████████████████████| 625/625 [15:16<00:00,  1.47s/it]


Train Loss : 0.6933 | Train Acc : 0.4994


 91%|████████████████████████████████████▎   | 568/625 [14:55<01:29,  1.58s/it]


KeyboardInterrupt: 

In [ ]:
# Test Acc

def test_loop(model,test_loader):
  model.eval()

  correct = 0
  total = 0

  with torch.no_grad():
    for images,labels in test_loader:

      images = images.to(device)
      labels = labels.to(device)

      logits = model(images)

      predictions = torch.round(torch.sigmoid(logits))

      correct += (predictions == labels).sum().item()
      total += labels.size(0)

  epoch_acc = correct / total

  print(f"Test Acc : {epoch_acc:.4f}")



test_loop(model,test_loader)